# Initialization

## Path configuration

In [ ]:
# Path configuration
import sys
from pathlib import Path

# Navigate up from 'src/notebooks/' to the project root directory
PROJECT_ROOT = Path("..").resolve().parent
DATA_DIR = PROJECT_ROOT / "data"
DOCS_DIR = PROJECT_ROOT / "docs"
SRC_DIR = PROJECT_ROOT / "src"

# Add src to python path for imports
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

# Enable automatic reloading (avoid restarting kernel when editing internal modules)
%reload_ext autoreload
%autoreload 2

## Imports

In [ ]:
from typing import Tuple

import pandas as pd
import numpy as np

import plotly.io as pio

from sklearn.model_selection import train_test_split

from utils.plots import plot_actuarial_profile, plot_interaction_heatmap, plot_exposure_by_claim_boxplot

In [ ]:
# Set renderer for plots
pio.renderers.default = "plotly_mimetype+notebook_connected"

# Data Integrity & Preprocessing

After loading the raw data we split it right away into training, validation and test sets. All data preprocessing and exploratory data analysis (EDA) will use only the training set to avoid any leakage from the validation and test sets.

Since the data set contains car insurance claims it is expected that the target column `ClaimNb` has many zeroes and a long tail. To ensure the target mean frequency is consistent across the training, validation, and test partitions, we apply stratified sampling to the split. To that end we use a modified version of `ClaimNb` with a capping on the long tail. This ensures statistical balance while maintaining the integrity of rare high-count events in the target variable.

In [ ]:
# Load raw data (publicly available at https://www.kaggle.com/datasets/floser/french-motor-claims-datasets-fremtpl2freq)
df_raw = pd.read_csv(DATA_DIR / "freMTPL2freq.csv")

In [ ]:
def train_val_test_split(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    # Define stratification column
    df["stratify_tmp"] = df["ClaimNb"].clip(upper=2)

    # Take 20% for testing
    df_train_val, df_test = train_test_split(df, test_size=0.2, random_state=13, stratify=df["stratify_tmp"])

    # Split the remaining 80% into training and validation
    # (taking 25% of the 80% gives us a 60/20/20 overall split)
    df_train, df_val = train_test_split(
        df_train_val, test_size=0.25, random_state=13, stratify=df_train_val["stratify_tmp"]
    )

    # Drop temporary stratification column
    df_train.drop(columns="stratify_tmp", inplace=True)
    df_val.drop(columns="stratify_tmp", inplace=True)
    df_test.drop(columns="stratify_tmp", inplace=True)

    # Print number of rows of each chunk
    print(f"Training rows:   {len(df_train)}")
    print(f"Validation rows: {len(df_val)}")
    print(f"Test rows:       {len(df_test)}")

    return df_train, df_val, df_test

In [ ]:
df_raw_train, df_raw_val, df_raw_test = train_val_test_split(df_raw)

## Data cleaning

The following function applies all cleaning and preprocessing to the raw training data. The remaining of the section is devoted to explaining all the decisions we made in this regard.

In [ ]:
def clean_raw_data(df_raw: pd.DataFrame) -> pd.DataFrame:
    df = df_raw.copy()

    # Set IDpol to integer type
    df["IDpol"] = df["IDpol"].astype(int)

    # Set VehAge to str type
    df["VehPower"] = df["VehPower"].astype(str)

    # Clip vehicle age
    df["VehAge"] = df["VehAge"].clip(upper=25)

    # Clip BonusMalus
    df["BonusMalus"] = df["BonusMalus"].clip(upper=150)

    # Clip number of claims
    df["ClaimNb"] = df["ClaimNb"].clip(upper=3)

    # Floor exposure (for offset and frequency calculations only)
    df["exposure_stable"] = df["Exposure"].clip(lower=1 / 12)

    return df


## Explaining cleaning choices

After initial exploration of the data we concluded that:

  - There are no null or unknown values that need treatment

  - `IDpol` can be safely converted to integer type

  - All categorical columns (`Area`, `VehBrand`, `VehGas`, `Region`) do not require any treatment since they have low cardinalities

  - `VehPower` transformed to string type so that models treat it as a category. Numerical order doesn't show a pattern with respect to claim frequency that justifies treating it differently.

  - `DrivAge` numerical column does not require treatment (all values are already between 18 and 100)

  - `VehAge` requires minimal treatment: we clip any value above 25 years old effectively creating a 25+ age (only 0.45% of the rows affected) that will account for all old cars.

  - `Density` numerical column does not need any treatment

  - The columns requiring more careful treatment are:

    - `ClaimNb`

    - `Exposure`

    - `BonusMalus` 

We shall include below the analysis and profiling that lead to the choices for these 3 columns (analysis for other columns is not included in this section but they are discussed in more depth during exploratory data analysis).

### ClaimNb

This is obviously a very relevant metric because it is the target value that our models will predict. Let us understand how it distributes:

In [ ]:
df_raw_train["ClaimNb"].value_counts().sort_index()

In [ ]:
print(
    f"Percentage of policies with no claims: {(len(df_raw_train[df_raw_train['ClaimNb'] == 0]) / len(df_raw_train) * 100):.2f}%"
)

We have a zero inflated distribution where ~95% of the policies have no claims. The profile represents a classic car insurance distribution where a vast majority of the policies have no claims or a small number of them. That justifies considering Poisson like distributions to model them (this will be explored more extensively later on).

However, for a Poisson distribution with that many zeroes, drivers with many claims per year are a "statiscal black swan". We shall make our models focus on the underlying characteristics that lead to claims in general. In order to prevent them over-weighting features of these extreme outliers, we propose clipping the data to 3 or 4 claims maximum.

The expected effect is introducing a bit of bias to reduce variance drastically, improving the signal-to-noise ratio. A high number of claims can exert disproportionate influence on the loss function, potentially leading to over-fitting to volatility rather than learning the risk. By capping the target, we align the model's objective with the prediction of typical claim frequency, ensuring more stable and generalized results across the test set.

In order to make a final decision we shall measure the damage of clipping in terms of data loss:

In [ ]:
def compute_claims_clipping_damage(df: pd.DataFrame, clipval: int) -> None:
    lost_claims = df["ClaimNb"].sum() - df["ClaimNb"].clip(upper=clipval).sum()
    percent_lost = (lost_claims / df["ClaimNb"].sum()) * 100

    print(f"Claims clipping damage (clip={clipval}):")
    print(f"  Total claims removed by clipping: {lost_claims}")
    print(f"  Percentage of total claim volume lost: {percent_lost:.4f}%")
    print()


compute_claims_clipping_damage(df_raw_train, clipval=3)
compute_claims_clipping_damage(df_raw_train, clipval=4)

**Profile:** A "zero-inflated" distribution where ~95% of policies have no claims.

**Treatment:** Clip at 3

**Logic:** The jump from 3+ to 4+ claims represents less than 0.05% of the data so we choose the option that helps the most at reducing overdispersion. This makes the data more suitable for models based on Poisson distributions.

**Business interpretation:** This profile represents a classic car insurance distribution where a vast majority of the policies have no claims or a small number of them. That justifies considering Poisson like distributions.

### Exposure

`Exposure` is another very relevant column since together with `ClaimNb` it determines the claim frequency. Any Poisson based distribution model predicting number of claims shall use it as an offset.

The main risk comes from very short exposures that make frequency explode. Our treatment will aim at taming that effect.

In [ ]:
df_raw_train["Exposure"].describe()

In [ ]:
df_raw_train["Exposure"].hist(bins=24)  # Each bin accounts for 1 month

In [ ]:
print(
    "Drivers with an exposure longer than a year (renewals): "
    f"{(len(df_raw_train[df_raw_train['Exposure'] > 1.0]) / len(df_raw_train) * 100):.2f}%"
)
print(
    "Drivers with an exposure shorter than a month (short term policies): "
    f"{(len(df_raw_train[df_raw_train['Exposure'] < 1 / 12]) / len(df_raw_train) * 100):.2f}%"
)

**Profile:** Significant bimodal distribution with peaks at <1 month and 12 months. ~17% of the data is short-term (<1 month). Only 0.19% of the data corresponds to drivers with exposures between 1 and 2 years.

**Treatment:** Floor at 1/12 (i.e. 1 month) for later frequency calculations; will be used as an offset/weighting for modeling.

**Logic:** Dropping 17% of the data would introduce survivorship bias. Flooring prevents "frequency explosions" (where 1 claim in 3 days appears as 100+ claims/year), allowing the models to learn from the entire population safely.

**Business interpretation:** Policies with less than 1 month exposure can be new business cancellations or cooling-off periods and hence have a different nature than policies with more exposure that represent an average customer. This will be addressed in more detail in the EDA.

### BonusMalus

Finally, `BonusMalus` is known to be a strong indicator for claim frequency in the car insurance industry. Since it is rated at a 50-350 scale but the break even (change from bonus to malus) is at 100 we shall treat it with care. Similarly to `ClaimNb`, we want to control for extreme outliers with very high malus that can add too much variance and deviate us from the main goal of modeling for most general type of claims.

In [ ]:
df_raw_train["BonusMalus"].describe()

In [ ]:
df_raw_train["BonusMalus"].hist(bins=50)

In [ ]:
print(
    "Drivers with perfect bonus (50): "
    f"{(len(df_raw_train[df_raw_train['BonusMalus'] == 50]) / len(df_raw_train) * 100):.2f}%"
)
print(
    f"Drivers with malus (>100): {(len(df_raw_train[df_raw_train['BonusMalus'] > 100]) / len(df_raw_train) * 100):.2f}%"
)
print(
    "Drivers with high malus (>150): "
    f"{(len(df_raw_train[df_raw_train['BonusMalus'] > 150]) / len(df_raw_train) * 100):.2f}%"
)

**Profile:** 56.73% of the data is pinned at the minimum score of 50. Only 1.14% of the data exists in the Malus zone (>100).

**Treatment:** Clip at 150

**Logic:** This preserves the high-risk signal of the malus drivers while preventing the 0.03% of extreme outliers from exerting undue leverage on the Poisson coefficients

**Business interpretation:** Most drivers have a perfect track record (50 bonus) and very few have malus. We shall see later that, as expected, this has a strong correlation with claim frequency

# Exploratory Data Analysis

We perform the exploratory data analysis on the cleaned training set that we will use for modeling:

In [ ]:
df_train = clean_raw_data(df_raw_train)

## Analysis

### BonusMalus

In [ ]:
plot_actuarial_profile(df_train, "num", "BonusMalus", bins=[49, 50, 75, 100, 125, 150])

The trend shows a sharp increase once we reach the malus zone. Creating an `is_malus` feature can help models discover this change.

### Driver Age

In [ ]:
plot_actuarial_profile(df_train, "num", "DrivAge", bins=[17, 25, 45, 65, 80, 100])

Again we see a hook-shaped trend. Here young drivers are detected to be significantly more prone to claims than more mature drivers. We can signal this fact to the models by creating an `is_young_driver` feature.

### Vehicle Age

In [ ]:
plot_actuarial_profile(df_train, "num", "VehAge", bins=[-1, 0, 5, 10, 15, 20, 25])

Similarly to young drivers, we can see how brand new cars (`VehAge`=0) get a much higher frequency of claims. After that, claims decline progressively as the vehicle gets older. Again, creating an `is_new_car` feature can be a useful signal for the models.

### Density

In [ ]:
plot_actuarial_profile(df_train, "num", "Density", bins=[0, 10, 100, 1000, 10000, 100000])

Note that we plotted `Density` at log scale and the frequency trend follows a fairly linear shape, especially for the most common densities (10 to 10000). In view of this fact we will create a new feature `log_density` to be used in our models.

### Vehicle Power / Gas / Brand

In [ ]:
for feature in ["Power", "Gas", "Brand"]:
    plot_actuarial_profile(df_train, "cat", f"Veh{feature}", None)

- For `VehPower` there are no major frequency differences among them. This is specially true for the most frequent ones (labels 4 to 7). It is not expected that this feature has much importance in the models unless there are hidden interactions with other features.

- `VehGas` has only two possible labels and both have similar levels of exposure and claim frequency. Similarly with what happens with power there is no direct evidence that this feature might be relevant at first glance.

- As for `VehBrand` there are no major differences but one of the most common brands, B12, is clearly the one with highest claim frequency. We shall see how much predictive power has that particular brand when modeling.

### Area / Region

In [ ]:
plot_actuarial_profile(df_train, "cat", "Area", None)
plot_actuarial_profile(df_train, "cat", "Region", None)

- `Area` seems to present a somewhat linear increase as we move from A to F. This may indicate some correlation with other metrics, possibly `log_density`. We shall explore it further below.

- `Region` does not present an obvious pattern or signal. There is no region with a lot of exposure that has a clearly higher or lower frequency of claims compared to other regions. We shall see how models handle region R94 since it has highest frequency but very little exposure.

In [ ]:
def area_logdensity_correlation(df: pd.DataFrame) -> float:
    dict_aux = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6}
    ps_area = df["Area"].apply(lambda x: dict_aux[x])
    ps_logdensity = np.log(df["Density"])
    correlation = np.corrcoef([ps_area, ps_logdensity])[0, 1]

    return correlation


print(f"Correlation between Area and log_density: {area_logdensity_correlation(df_train):.2f}")

A correlation of 0.97 confirms that `Area` is a proxy of `log_density`, so we can safely drop it from the models since it will not add any extra relevant information. Of the two, it is better to keep `log_density` since it is continuous and more granular.

### DrivAge-BonusMalus interaction

In [ ]:
plot_interaction_heatmap(df_train, "DrivAge", [17, 25, 45, 65, 80, 100], "BonusMalus", [49, 50, 75, 100, 125, 150])

From univariate analysis we knew that young drivers and high malus have higher frequencies, so it was expected for the combination *young+malus* to have higher frequency. However, here we are seeing another kind of combination that is even riskier: *old+malus*. This interaction suggests that the low-risk benefit of an older driver is completely offset by the behavioral risk signaled by a high malus score. 

A model that is able to capture such kind of interactions, like gradient boosted decision trees, should perform well on the dataset.

### VehAge-BonusMalus interaction

In [ ]:
plot_interaction_heatmap(df_train, "VehAge", [-1, 0, 5, 10, 15, 20, 25], "BonusMalus", [49, 50, 75, 100, 125, 150])

Here we see the exact same pattern that we saw for older driver age: older cars are not a risk on themselves, but they become a very high one one combined with malus.

### VehAge-DrivAge interaction

In [ ]:
plot_interaction_heatmap(df_train, "VehAge", [-1, 0, 5, 10, 15, 20, 25], "DrivAge", [17, 25, 45, 65, 80, 100])

After analyzing `BonusMalus` against `DrivAge` and `VehAge` and uncovering risk pockets that we wouldn't expect from univariate analysis alone, we see here that when we compare driver and vehicle age we do get what one would expect from the univariate analysis: young drivers in new cars are a higher risk combination.

Still, though, it is interesting to see how here the highest frequency is 0.385 when old people or old cars combined with high malus is in the order of 0.8 to 0.9, a much higher risk.

### Exposure by ClaimNb

In [ ]:
plot_exposure_by_claim_boxplot(df_train)

Demonstrating the relationship between `Exposure` and `ClaimNb`, justifying the use of the latter as an offset/weight:
  
  - Median shift: median exposure for 0 claims is much lower than when there are actual claims
  
  - Visual proof than the longer a policy is active (higher exposure) the more opportunity for a claim to occur
  
  - The first quartile of the 0 claims box highlights the large volume of short-term policies

## Feature engineering

In view of the EDA we add some binary helper columns that represent actuarial value:
  
  - `is_malus`: In many markets, being in the malus (penalty) zone triggers a different risk profile than just being a bad driver in the bonus zone.

  - `is_young_driver`: Drivers aged 18–25 represent a structural discontinuity in risk, often subject to different underwriting rules, second-driver exclusions, and higher behavioral volatility than the general population.

  - `is_new_car`: Brand-new cars often have a specific risk profile (higher frequency due to meticulous owners or lower due to better safety tech).

  - `is_short_term`: Very short policies are often associated with temporary or high-churn risks that act differently than standard annual policies.

We also create a new `log_density` feature, expected to have a linear relationship with the frequency of claims.

In [ ]:
def add_new_features(df: pd.DataFrame) -> pd.DataFrame:
    # Actuarial features
    df["is_malus"] = (df["BonusMalus"] > 100).astype(int)
    df["is_young_driver"] = (df["DrivAge"] <= 25).astype(int)
    df["is_new_car"] = (df["VehAge"] == 0).astype(int)
    df["is_short_term"] = (df["Exposure"] < 1 / 12).astype(int)

    # Log(Density)
    df["log_density"] = np.log(df["Density"])

    return df

## Data preparation

Finally, we make the data ready for modeling. To that end we clean `df_val` and `df_test` in the same way we cleaned `df_train` and we make sure that all three sets have the new features.

In [ ]:
# Clean validation and test data sets
df_val = clean_raw_data(df_raw_val)
df_test = clean_raw_data(df_raw_test)

In [ ]:
# Add the new features
df_train = add_new_features(df_train)
df_val = add_new_features(df_val)
df_test = add_new_features(df_test)

In [ ]:
# Store cleaned files for future use in modeling
df_train.to_csv(DOCS_DIR / "train.csv", index=False)
df_val.to_csv(DOCS_DIR / "val.csv", index=False)
df_test.to_csv(DOCS_DIR / "test.csv", index=False)